In [1]:
import pandas as pd

# Load your DataFrame (example)
df = pd.read_csv("merged_all_filtered.csv")

# Drop columns that contain "13" in their name
df_cleaned = df.loc[:, ~df.columns.str.contains("13")]

# Save to a new file if needed
df_cleaned.to_csv("merged_all_filtered_no13.csv", index=False)


### Normalization

In [2]:
import numpy as np
import pandas as pd
import re

# Load your DataFrame
df = pd.read_csv("merged_all_filtered_no13.csv")

# Base columns to normalize
base_columns = ['ALLSKY_SFC_SW_DWN','ALLSKY_KT', 'CLOUD_AMT', 'PRECTOTCORR']

# Start with lat and lon in the new DataFrame
normalized_columns = {
    'lat': df['lat'],
    'lon': df['lon']
}

# Normalize matching monthly columns
for col_base in base_columns:
    pattern = re.compile(rf'^{col_base}_\d{{6}}$')  # Matches e.g. ALLSKY_SFC_SW_DWN_202001
    matching_cols = [col for col in df.columns if pattern.match(col)]

    for col in matching_cols:
        min_val = df[col].min()
        max_val = df[col].max()
        if max_val - min_val != 0:
            normalized_columns[col] = ((df[col] - min_val) / (max_val - min_val)).round(5)
        else:
            normalized_columns[col] = df[col].round(5)

# Create the final DataFrame
normalized_df = pd.DataFrame(normalized_columns)

# Save to CSV
normalized_df.to_csv("all_data_normalized.csv", index=False)


In [3]:
import pandas as pd
features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_KT', 'CLOUD_AMT', 'PRECTOTCORR']
unique_counts = {}
df = pd.read_csv("all_data_normalized.csv")
for feature in features:
    # Get all monthly columns for this feature
    cols = [col for col in df.columns if col.startswith(feature)]
    
    # Concatenate all values into one Series
    combined = pd.concat([df[col] for col in cols])
    
    # Count unique values across all months
    unique_counts[feature] = combined.nunique()

# Show result
for feat, count in unique_counts.items():
    print(f"{feat} → {count} unique values across months")


ALLSKY_SFC_SW_DWN → 27156 unique values across months
ALLSKY_KT → 3889 unique values across months
CLOUD_AMT → 28685 unique values across months
PRECTOTCORR → 22741 unique values across months


## Monthly scores

In [4]:
import pandas as pd

# Load the updated dataset with ALLSKY_KT included
df = pd.read_csv("all_data_normalized.csv")

# Define the suitability function with all four parameters
def calculate_suitability(row, w0=0.8, w1=0.3, w2=-0.1, w3=-0.1):
    return (
        w0 * row['ALLSKY_SFC_SW_DWN'] +
        w1 * row['ALLSKY_KT'] +
        w2 * row['CLOUD_AMT'] +
        w3 * row['PRECTOTCORR']
    )

# Prepare a DataFrame to store the monthly scores
monthly_scores = df[['lat', 'lon']].copy()

# Loop through each month in 2020-2023
for year in range(2020, 2024):
    for month in range(1, 13):
        ym = f"{year}{month:02d}"
        try:
            score = df.apply(lambda row: calculate_suitability({
                'ALLSKY_SFC_SW_DWN': row[f'ALLSKY_SFC_SW_DWN_{ym}'],
                'ALLSKY_KT': row[f'ALLSKY_KT_{ym}'],
                'CLOUD_AMT': row[f'CLOUD_AMT_{ym}'],
                'PRECTOTCORR': row[f'PRECTOTCORR_{ym}']
            }), axis=1)
            monthly_scores[f'score_{ym}'] = score.round(5)
        except KeyError:
            print(f"Missing data for month {ym} — skipping.")

# Identify the score columns (those that start with "score_")
score_columns = [col for col in monthly_scores.columns if col.startswith("score_")]

# Calculate the average score across all 48 months
monthly_scores['avg_score'] = monthly_scores[score_columns].mean(axis=1).round(5)

# Save the result
output_path = "monthly_suitability_scores_with_avg.csv"
monthly_scores.to_csv(output_path, index=False)

output_path


'monthly_suitability_scores_with_avg.csv'

In [5]:
# Extract only the relevant columns
df = pd.read_csv("all_data_normalized.csv")

avg_score_df = monthly_scores[['lat', 'lon', 'avg_score']]

# Save to a new CSV file
output_path_avg_only = "lat_lon_avg_score.csv"
avg_score_df.to_csv(output_path_avg_only, index=False)

output_path_avg_only


'lat_lon_avg_score.csv'